<a href="https://colab.research.google.com/github/OdehDan/IBM_HR_Attrition_Analysis/blob/main/HR_Employee_Attrition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**GROUP 02**

**DATASET** - IBM HR ANALYTICS EMPLOYEE ATTRITION.

**DEPTH TRACK** - CLUSTERING DEPTH

**TEAM MEMBER NAMES:**

**1)ESSANGENYI BRIGHT VICTOR**

**2)DANIEL OFUKOWOICHO ODEH**

**3)UWEH BLESSING**

**4)SUNDAY GLORIA AUDU**

**5)RACHEL OLUWATOBILOBA ODEJOBI**

**6)ADEYEMO OREOLUWA VICTORIA**

**7)KEHINDE KANYINSOLA TAIWO**

**8)UBONG KINGSLEY UFOT**

**9)OGUNSOLA IYABO NGOZI**

**10)AMUSA QUDUS KEHINDE**

**PART 1 - DATA FOUNDATION**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, mean_absolute_error, r2_score, classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Install
!pip install kagglehub --quiet

# Import
import kagglehub
import pandas as pd
import os

# Download
path = kagglehub.dataset_download('pavansubhasht/ibm-hr-analytics-attrition-dataset')

# Load
files = os.listdir(path)
df = pd.read_csv(os.path.join(path, files[0]))

print(f"Dataset loaded successfully: {df.shape}")

In [ ]:
df.head()

In [ ]:
conn = sqlite3.connect("data.db")
df.to_sql("hr_attrition", conn, if_exists="replace", index=False)

In [ ]:
pd.read_sql_query(
    """
    SELECT EducationField, AVG(MonthlyIncome) AS AvgMonthlyIncome
    FROM hr_attrition
    GROUP BY EducationField
    Having AVG(MonthlyIncome) > 5000
    ORDER BY AvgMonthlyIncome DESC;
    """,
    conn
)

From the above SQL query, the Education Field with the highest Average Monthly Salary greater than 10,000 is Marketing followed by Human Resources, then Medical, followed by Life Sciences, then Other with the least being Technical Degree.

In [ ]:
pd.read_sql_query("""
    SELECT
        Attrition,
        ROUND(AVG(MonthlyIncome), 2)     AS avg_monthly_income,
        ROUND(AVG(YearsAtCompany), 2)    AS avg_years_at_company,
        ROUND(AVG(Age), 2)               AS avg_age,
        COUNT(*)                          AS headcount
    FROM hr_attrition
    GROUP BY Attrition
""", conn)

This query tells us how people attrited based on their monthly salary, average years at the company, age and it calculates the number of employees that left and those that stayed.

In [ ]:
pd.read_sql_query(
    """
    SELECT JobRole, AVG(YearsSinceLastPromotion) AS AvgYearsSincePromotion
    FROM hr_attrition
    GROUP BY JobRole
    ORDER BY AvgYearsSincePromotion DESC
    LIMIT 5;
    """,
    conn
)

From the above SQL query, we can determine the roles that are waiting longest for promotion on an average. Based on the query above, the longest waiting is the Manager, followed by Research Director with the least being Manufacturing Director. Some of these roles that have the longest years is because they are already at top roles.

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

In [ ]:
plt.figure(figsize=(10,8))
sns.histplot(df["Attrition"])
plt.xlabel("Attrition")
plt.ylabel("Count")
plt.title("Frequency Distribution Of Attrition")
plt.grid(True, alpha=0.5)
plt.show()

In [ ]:
plt.figure(figsize=(10,8))
sns.histplot(df["TotalWorkingYears"], kde=True)
plt.xlabel("YearsAtCompany")
plt.ylabel("Count")
plt.title("Frequency Distribution Of Total Working Years At Company")
plt.grid(True, alpha=0.5)
plt.show()

In [ ]:
plt.figure(figsize=(10,8))
sns.histplot(df["MonthlyIncome"], kde=True)
plt.xlabel("Monthly Income")
plt.ylabel("Count")
plt.title("Frequency Distribution Of Monthly Income of Employees")
plt.grid(True, alpha=0.5)
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
sns.histplot(df["OverTime"])
plt.xlabel("OverTime")
plt.ylabel("Count")
plt.title("Frequency Distribution Of OverTime")
plt.grid(True, alpha=0.5)
plt.show()

In [ ]:
df["NumCompaniesWorked"].replace(0, df["NumCompaniesWorked"].median())

We replaced the missing zeros in NumOfCompaniesWorked column with median because after investigating the zeros value we found out some abnormal patterns like an employee who is 38 and with Job Level 3 having never worked at any company is not realistic.

Also, we checked other colums concerning the zeros and it was justified.

In [ ]:
df=df.drop(["EmployeeCount", "EmployeeNumber", "Over18", "StandardHours"], axis=1)

We dropped the above columns due to the following reasons. We dropped Employee Count, Over18 and StandardHours because it had a constant value and it would not add anything meaningful to our prediction. We also dropped EmployeeNumber column because it is just like an employee ID and it is not useful for our predictions too.


**PART 2 - FEATURE ENGINEERING AND ENCODING**

In [ ]:
df["IncomePerExperienceYear"] = df["MonthlyIncome"] / (df["TotalWorkingYears"] + 1)

We created another feature called Income Per Experience Year where we employed the ratio year. It calculates the monthly Income of an employee in relation to the Total Working Years. The +1 added is for employees with zero total working years so as to prevent error in the calculation. It measures if experienced workers are being paid properly.

In [ ]:
df['TenureRatio'] = df['YearsAtCompany'] / (df['TotalWorkingYears'] + 1)

This new feature engineered represents or calculates the years spent at a company with in relation to the total working years.

In [ ]:
df["WorkingExperience"] = pd.cut(df["TotalWorkingYears"], bins=[0,3,7,15,40], labels=["L1_Entry", "L2_Mid", "L3_Experienced", "L4_Veteran"], include_lowest=True)

We created a new feature for my dataset called Working Experience where we employed the Binning method. It groups the Total Working Years into four categories and assigns them to their respective labels.
0-3 = Entry Level,
3-7 = Mid Level,
7-15 = Experienced,
15-40 = Veteran.

In [ ]:
df["MonthlyIncome_per_JobLevel"] = df["MonthlyIncome"] / df["JobLevel"]

We created another feature where we combined two columns to an interaction column. It reveals compensation fairness relative to each employee's Job Level.

In [ ]:
le = LabelEncoder()
df["WorkingExperience"] = le.fit_transform(df["WorkingExperience"])

We used Label Encoder for my Working Experience column. This is beause it encodes based on rank when something is greater than the order and Working Experience had attributes that was needed to be ranked.

In [ ]:
columns_to_encode = ["Attrition", "BusinessTravel", "Department", "EducationField", "Gender", "JobRole", "MaritalStatus", "OverTime"]

We used One Hot Encoder for the remaining categorical columns. This is because firstly, we need to convert them to numbers and because the attributes we have in the other columns do not need to be ranked.

In [ ]:
df_onehot=pd.get_dummies(df[columns_to_encode], drop_first=True)

In [ ]:
df = df.drop(df[columns_to_encode], axis=1)

In [ ]:
df = pd.concat([df.reset_index(drop=True), df_onehot.reset_index(drop=True)], axis=1)

We dropped the initial columns because the were categorical and we just encoded them. So we concatenated the encoded columns together with the numerical features so as to have them all numerical.

In [ ]:
X = df.drop(["Attrition_Yes"], axis=1).astype(float)
Y = df["Attrition_Yes"].astype(float)

In [ ]:
sc = StandardScaler()
X_scaled = sc.fit_transform(X)

We used Standard scaler because the mean and standard deviation were far from each other and I wanted each column to have a mean 0 and standard deviation 1.

**PART 3 - CLUSTERING ANALYSIS**

In [ ]:
#Elbow Method
inertias = []
k_range = range(2,11)

for k in k_range:
  kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
  kmeans.fit(X_scaled)
  inertias.append(kmeans.inertia_)
  print(f"K = {k}: Inertia = {kmeans.inertia_:.2f}")

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(k_range, inertias, marker="o", linestyle="-")
plt.xlabel("Number Of Clusters(k)")
plt.ylabel("Inertia")
plt.title("Elbow Method for Optimal K")
plt.grid(True)
plt.show()

In [ ]:
#Silhouette score
from sklearn.metrics import silhouette_score
silhouette_scores = []
k_range = range(2,11)

for k in k_range:
  kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
  cluster_labels = kmeans.fit_predict(X_scaled)

  silhouette_avg = silhouette_score(X_scaled, cluster_labels)
  silhouette_scores.append(silhouette_avg)
  print(f"K = {k} : Silhouette Score = {silhouette_avg}")

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(k_range, silhouette_scores, marker="o", linestyle="-")
plt.xlabel("Number Of Clusters(k)")
plt.ylabel("Silhouette Scores")
plt.title("Silhouette Scores for K")
plt.grid(True)
plt.show()

In [ ]:
# Building final KMeans
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)

We picked my K value to be 4 based on the elbow method. Although the silhouette score suggested 2, we chose 4 because it provides more granular and actionable employee personas that are more useful for HR decision making. The elbow method also supported K=4 as the point of maximum curvature.

In [ ]:
df["kmeans_clusters"] = kmeans_labels
df["kmeans_clusters"].value_counts()

In [ ]:
df.groupby("kmeans_clusters").agg({
    "MonthlyIncome" : "mean",
    "IncomePerExperienceYear" : "mean",
    "YearsWithCurrManager" : "mean",
    "YearsSinceLastPromotion" : "mean",
    "Age" : "mean",
    "JobLevel" : "mean",
    "NumCompaniesWorked" : "mean",
    "MonthlyIncome_per_JobLevel" : "mean",
    "OverTime_Yes" : "mean"
}).round(2)


Cluster 0 represents Tenured High Level Professionals. Employees in this category are the oldest among the clusters with the most paid average monthly income of $14968.15. They have the most years spent with their current manager, highest Job Level coupled with Income per Experience Year. They spent reasonable time working Over time. They are second to the last contributors to the different clusters.

Cluster 1 represents the Balanced Career Progressors. The employees here have a very decent monthly income of about $6781.26 on average which is the second highest. They are middle aged with a good number of working years with their current manager. Coupled with the fact that their Job Level is decent, they work little or medium overtime.

Cluster 2 represents the Low Income Early Career Drifters. The employees here are the youngest with the lowest Monthly Income and Job Level. Although their Years since last promotion, Income per experience year and Over time are very okay. They tend to leave the company more.

Cluster 3 represents the Experienced but Undervalued Staff. Employees here are middle aged but they are not compensated well enough with the second to the least Monthly salary. Their Job Level is low but the years with their current manager and years since last promotion is really good, meaning they are undervalued.

In [ ]:
# Pulling out a few representative customers row from each cluster

# Map cluster numbers to your persona names
persona_names = {
    0: 'Tenured High Level Professionals',
    1: 'Balanced Career Progressors',
    2: 'Low Income Early Career Drifters',
    3: 'Experienced but Undervalued Staff'
}

df['Persona'] = df['kmeans_clusters'].map(persona_names)

# Define features used for profiling
profile_cols = ['MonthlyIncome', 'IncomePerExperienceYear', 'YearsWithCurrManager',
                'YearsSinceLastPromotion', 'Age', 'JobLevel', 'JobSatisfaction',
                'JobInvolvement', 'OverTime_Yes']

# Display columns for representative rows
display_cols = ['Age', 'MonthlyIncome', 'IncomePerExperienceYear', 'JobLevel',
                'YearsWithCurrManager', 'YearsSinceLastPromotion',
                'JobSatisfaction', 'OverTime_Yes', 'Persona']

# Pull 3 representative rows per persona
for cluster in sorted(df['kmeans_clusters'].unique()):

    cluster_df = df[df['kmeans_clusters'] == cluster].copy()
    cluster_mean = cluster_df[profile_cols].mean()

    cluster_df['distance_from_mean'] = cluster_df[profile_cols].apply(
        lambda row: ((row - cluster_mean) ** 2).sum() ** 0.5, axis=1
    )

    representative = cluster_df.nsmallest(3, 'distance_from_mean')[display_cols]
    attrition_rate = round(cluster_df['Attrition_Yes'].mean() * 100, 2)

    print(f"\n{'='*65}")
    print(f"Cluster {cluster} — {persona_names[cluster]}")
    print(f"Size: {len(cluster_df)} employees | Attrition Rate: {attrition_rate}%")
    print(f"{'='*65}")
    print(representative.to_string(index=True))

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

In [ ]:
plt.figure(figsize=(9, 6))
plt.scatter(
    X_pca[:, 0], X_pca[:, 1],
    c=kmeans_labels, cmap="viridis", alpha=0.6, s=15
)
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.title("Employee Clusters (K-Means)")
plt.colorbar(label="Cluster")
plt.show()

**CLUSTERING DEPTH**

In [ ]:
# Using K=2 for clustering depth
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
kmeans_labels_2 = kmeans.fit_predict(X_scaled)

In [ ]:
df["kmeans_clusters_2"] = kmeans_labels_2
df["kmeans_clusters_2"].value_counts()

In [ ]:
df.groupby("kmeans_clusters_2").agg({
    "MonthlyIncome" : "mean",
    "IncomePerExperienceYear" : "mean",
    "YearsWithCurrManager" : "mean",
    "YearsSinceLastPromotion" : "mean",
    "Age" : "mean",
    "JobLevel" : "mean",
    "NumCompaniesWorked" : "mean",
    "MonthlyIncome_per_JobLevel" : "mean",
    "OverTime_Yes" : "mean"
}).round(2)

  Cluster 1 represents the Developing Workforce Employees. The employees in this cluster are in their growth phase. They comprise of people with lesser age, lower monthly income, lesser years with their current manager, lesser income per experience year and Overtime. Employees here are mostly in their early stages building their carrers and starting from lower ranks.

  Cluster 0 represents Established Senior Professionals. Employees here have a stron relationsip wit teir current manager, they are older, they work more overtime, having higher monthly income compared to cluster 0 and have higher Job Level. They have stayed in the company longer.

In [ ]:
kmeans = KMeans(n_clusters=7, random_state=42, n_init=10)
kmeans_labels_3 = kmeans.fit_predict(X_scaled)

In [ ]:
df["kmeans_clusters_3"] = kmeans_labels_3
df["kmeans_clusters_3"].value_counts()

In [ ]:
df.groupby("kmeans_clusters_3").agg({
    "MonthlyIncome" : "mean",
    "IncomePerExperienceYear" : "mean",
    "YearsWithCurrManager" : "mean",
    "YearsSinceLastPromotion" : "mean",
    "Age" : "mean",
    "JobLevel" : "mean",
    "NumCompaniesWorked" : "mean",
    "MonthlyIncome_per_JobLevel" : "mean",
    "OverTime_Yes" : "mean"
}).round(2)

Cluster 0: Young Entry-Level (Low Overtime)
Lowest average age (approx. 33.5) and lowest Job Level (1.15). They have the lowest Monthly Income (~$3,168) but a solid Income Per Experience Year, indicating strong entry starting rates. Relatively low overtime (35%).

Cluster 5: Young Entry-Level (High Overtime / Low Turnover Risk)
Similar profile to Cluster 0 (Age ~33.5, Job Level 1.17, Income ~$3,104), but they work significantly fewer hours of overtime (24%) and have stable, recent management relationships.

Cluster 6: Youngest "Gig-Hoppers" (Highest Attrition Risk)
The youngest group (average age 30) with the lowest Job Level (1.08) and lowest Monthly Income ($2,626). They have high overtime (29%) and very low tenure with their current manager (1.66 years), marking them as a high-risk group for early turnover.

Cluster 2: Fast-Track Mid-Levels
Mid-thirties (Age ~36.6) with solid mid-management status (Job Level 2.31). They command an efficient income-to-level ratio ($2,948 per level) and exhibit balanced tenure metrics across the board.

Cluster 3: Mid-Level Veterans (Stagnant / Low Growth)
Slightly older mid-levels (Age ~38.5) but stuck at a lower Job Level (2.08) and lower Income Per Experience Year ($575). They have very low tenure with their current manager (2.41 years), indicating potential career stagnation.

Cluster 1: Promoted Senior Leaders (High Promotion Stagnation)
Late-thirties professionals (Age 38.6) with high Job Levels (2.33) and strong Monthly Income ($6,762). However, they have the highest time elapsed since their last promotion (4.63 years), making them prime candidates for a retention/progression review.

Cluster 4: Elite Executive Tier
The oldest (Age 45.8), highest-earning cluster by a massive margin ($16,539 monthly). They hold the highest Job Level(4.15) and Income Per Experience year(732.22). They have long managerial stability (6.18 years) and normal promotion cycles.

**PART 4 - CLASSIFICATION**

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X_scaled, Y, test_size=0.2, random_state=42, stratify=Y)

In [ ]:
# Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, Y_train)

In [ ]:
y_proba_lr = lr.predict_proba(X_test)[:, 1]

In [ ]:
y_pred_train = lr.predict(X_train)
y_pred_train[:5]

In [ ]:
results_df = pd.DataFrame({
    "Actual" : Y_train,
    "Predicted" : y_pred_train
})
display(results_df.head(10))

In [ ]:
y_pred_test = lr.predict(X_test)
y_pred_test[:5]

In [ ]:
results_df_test = pd.DataFrame({
    "Actual" : Y_test,
    "Predicted" : y_pred_test
})
display(results_df_test.head(10))

In [ ]:
feature_importance = pd.DataFrame({
    "Feature" : X.columns,
    "Importance" : lr.coef_[0]
}).sort_values(ascending=False, by="Importance")
display(feature_importance.head(10))

In [ ]:
from sklearn.metrics import classification_report,confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
print(classification_report(Y_train, y_pred_train))

In [ ]:
print(classification_report(Y_test, y_pred_test))

In [ ]:
print(f"Confusion Matrix(Test Set): ")
cm = confusion_matrix(Y_test, y_pred_test)
print(cm)

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="coolwarm")
plt.xlabel("Predicted Value")
plt.ylabel("Actual Value")
plt.title("Confusion Matrix for Logistic Regression")
plt.show()

**DECISION TREE CLASSIFIER**

In [ ]:
dt = DecisionTreeClassifier(max_depth=5, min_samples_split=20, min_samples_leaf=10, random_state=42)
dt.fit(X_train,Y_train)

In [ ]:
y_pred_dt = dt.predict(X_train)
y_pred_train[:5]

In [ ]:
y_pred_dt2 = dt.predict(X_test)
y_pred_test[:5]

In [ ]:
y_proba_dt = dt.predict_proba(X_test)[:, 1]

In [ ]:
print(classification_report(Y_train, y_pred_dt))

In [ ]:
print(classification_report(Y_test, y_pred_dt2))

In [ ]:
feature_importance = pd.DataFrame({
    "Feature" : X.columns,
    "importance" : dt.feature_importances_
}).sort_values("importance", ascending=False)
display(feature_importance.head())

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="coolwarm")
plt.xlabel("Predicted Value")
plt.ylabel("Actual Value")
plt.title("Confusion Matrix for Decision Tree Classifier")
plt.show()

**RANDOM FOREST CLASSIFIER**

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=20, min_samples_leaf=10, random_state=42, n_jobs=-1)
rf.fit(X_train, Y_train)

In [ ]:
y_pred_rf = rf.predict(X_train)
y_pred_rf[:10]

In [ ]:
y_pred_rf2 = rf.predict(X_test)
y_pred_rf2[:10]

In [ ]:
y_proba_rf = rf.predict_proba(X_test)[:, 1]

In [ ]:
rf_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False).head(10)

rf_importance

In [ ]:
print(classification_report(Y_train, y_pred_rf))

In [ ]:
print(classification_report(Y_test, y_pred_rf2))

In [ ]:
print("Confusion Matrix(Test Set): ")
cm3=confusion_matrix(Y_test, y_pred_rf2)
print(cm3)

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(cm3, annot=True, fmt="d", cmap="coolwarm")
plt.xlabel("Predicted Value")
plt.ylabel("Actual Value")
plt.title("Confusion Matrix for Random Forest Classifier")
plt.show()

In [ ]:
def evaluate_model(name, y_true, y_pred, y_prob):
  return{
      "model" : name,
      "accuracy" : accuracy_score(y_true, y_pred),
      "precision" : precision_score(y_true, y_pred),
      "recall" : recall_score(y_true, y_pred),
      "f1_score" : f1_score(y_true, y_pred),
      "roc_auc" : roc_auc_score(y_true, y_prob)
  }

In [ ]:
results = pd.DataFrame([
    evaluate_model("Logistic Regression", Y_test, y_pred_test, y_proba_lr),
    evaluate_model("Decision Tree", Y_test, y_pred_dt2, y_proba_dt),
    evaluate_model("Random Forest", Y_test, y_pred_rf2, y_proba_rf)
])
results.round(5)

For the classification techniques, we used:

1. Logistic Regression because it is simple, fast, and interpretable, making it useful for understanding how factors like overtime, income, or job satisfaction affect the likelihood of employees leaving. It is interpretable linear baseline; shows which features push the probability up or down.

2. Decision Tree because it can capture nonlinear relationships and decision rules in employee behavior. It is easy to visualize and explain, showing clear paths such as employees with low job satisfaction and high overtime are more likely to leave.

3. Random Forest because improves prediction accuracy by combining multiple decision trees. It reduces overfitting and handles complex patterns in HR data better, making it more reliable for predicting employee attrition compared to a single Decision Tree.

**IMBALANCE HANDLING - (LOGISTIC REGRESSION)**

In [ ]:
# Class weight
lr_weighted = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
lr_weighted.fit(X_train, Y_train)
y_pred_w = lr_weighted.predict(X_test)
y_prob_w = lr_weighted.predict_proba(X_test)[:, 1]

print("Logistic Regression + Class Weights:")
print(classification_report(Y_test, y_pred_w))

In [ ]:
#Threshold Tuning
thresholds=[0.5,0.4,0.3,0.2]
threshold_results=[]

for t in thresholds:
  y_pred_t = (y_proba_lr >= t).astype(int)
  threshold_results.append({
    "threshold": t,
    "precision": precision_score(Y_test, y_pred_t),
    "recall": recall_score(Y_test, y_pred_t),
    "f1": f1_score(Y_test, y_pred_t),
    })
pd.DataFrame(threshold_results).round(3)

In [ ]:
from sklearn.metrics import precision_recall_curve
pre, rec, thr = precision_recall_curve(Y_test, y_proba_lr)
plt.figure(figsize=(10,6))
plt.plot(rec, pre, marker="o", markersize=3)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
#SMOTE
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, Y_train_smote = smote.fit_resample(X_train, Y_train)

In [ ]:
Y_train_smote.value_counts()

In [ ]:
lr_smote = LogisticRegression(max_iter=1000)
lr_smote.fit(X_train_smote, Y_train_smote)

In [ ]:
y_pred_lr_smote = lr_smote.predict(X_test)
y_proba_lr_smote = lr_smote.predict_proba(X_test)[:, 1]

In [ ]:
print(classification_report(Y_test, y_pred_lr_smote))

In [ ]:
y_pred_threshold = (y_proba_lr >= 0.3).astype(int)

imbalance_results = pd.DataFrame([
    evaluate_model("Baseline LR", Y_test, y_pred_test, y_proba_lr),
    evaluate_model("LR + threshold 0.3", Y_test, y_pred_threshold, y_proba_lr),
    evaluate_model("LR + SMOTE", Y_test, y_pred_lr_smote, y_proba_lr_smote),
])

imbalance_results.round(3)

In [ ]:
feature_importance = pd.DataFrame({
    "Feature" : X.columns,
    "Importance" : lr.coef_[0]
}).sort_values(ascending=False, by="Importance")
display(feature_importance.head(10))

**IMBALANCE HANDLING - (RANDOM FOREST)**

In [ ]:
# Class weight
rf_weighted = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_split=20, min_samples_leaf=10, random_state=42, n_jobs=-1, class_weight="balanced")
rf_weighted.fit(X_train, Y_train)
y_pred_w_rf = rf_weighted.predict(X_test)
y_prob_w_rf = rf_weighted.predict_proba(X_test)[:, 1]

print("Random Forest + Class Weights:")
print(classification_report(Y_test, y_pred_w_rf))

In [ ]:
# Threshold tuning
thresholds=[0.5,0.4,0.3,0.2]
threshold_results=[]

for t in thresholds:
  y_pred_t = (y_proba_rf >= t).astype(int)
  threshold_results.append({
    "threshold": t,
    "precision": precision_score(Y_test, y_pred_t),
    "recall": recall_score(Y_test, y_pred_t),
    "f1": f1_score(Y_test, y_pred_t),
    })
pd.DataFrame(threshold_results).round(3)

In [ ]:
from sklearn.metrics import precision_recall_curve
pre, rec, thr = precision_recall_curve(Y_test, y_proba_rf)
plt.figure(figsize=(10,6))
plt.plot(rec, pre, marker="o", markersize=3)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# SMOTE (RANDOM FOREST)
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, Y_train_smote = smote.fit_resample(X_train, Y_train)

In [ ]:
Y_train_smote.value_counts()

In [ ]:
rf_smote = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)
rf_smote.fit(X_train_smote, Y_train_smote)

In [ ]:
y_pred_rf_smote = rf_smote.predict(X_test)
y_proba_rf_smote = rf_smote.predict_proba(X_test)[:, 1]

In [ ]:
print(classification_report(Y_test, y_pred_rf_smote))

In [ ]:
y_pred_threshold = (y_proba_rf >= 0.2).astype(int)

imbalance_results = pd.DataFrame([
    evaluate_model("Baseline RF", Y_test, y_pred_rf2, y_proba_rf),
    evaluate_model("RF + threshold 0.2", Y_test, y_pred_threshold, y_proba_rf),
    evaluate_model("RF + SMOTE", Y_test, y_pred_rf_smote, y_proba_rf_smote),
])

imbalance_results.round(3)

**IMBALANCE HANDLING - (DECISION TREE)**

In [ ]:
# Class weight
dt_weighted = RandomForestClassifier(max_depth=5, min_samples_split=20, min_samples_leaf=10, random_state=42, class_weight="balanced")
dt_weighted.fit(X_train, Y_train)
y_pred_w_dt = dt_weighted.predict(X_test)
y_prob_w_dt = dt_weighted.predict_proba(X_test)[:, 1]

print("Decision Tree + Class Weights:")
print(classification_report(Y_test, y_pred_w_dt))

In [ ]:
# Threshold Tuning
thresholds=[0.5,0.4,0.3,0.2]
threshold_results=[]

for t in thresholds:
  y_pred_t = (y_proba_dt >= t).astype(int)
  threshold_results.append({
    "threshold": t,
    "precision": precision_score(Y_test, y_pred_t),
    "recall": recall_score(Y_test, y_pred_t),
    "f1": f1_score(Y_test, y_pred_t),
    })
pd.DataFrame(threshold_results).round(3)

In [ ]:
from sklearn.metrics import precision_recall_curve
pre, rec, thr = precision_recall_curve(Y_test, y_proba_rf)
plt.figure(figsize=(10,6))
plt.plot(rec, pre, marker="o", markersize=3)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# SMOTE
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, Y_train_smote = smote.fit_resample(X_train, Y_train)

In [ ]:
dt_smote = DecisionTreeClassifier(max_depth=5, min_samples_split=20, min_samples_leaf=10, random_state=42)
dt_smote.fit(X_train_smote,Y_train_smote)

In [ ]:

dt_smote = DecisionTreeClassifier(max_depth=5, min_samples_split=20, min_samples_leaf=10, random_state=42)
dt_smote.fit(X_train_smote,Y_train_smote)

In [ ]:
y_pred_dt_smote = dt_smote.predict(X_test)
y_proba_dt_smote = dt_smote.predict_proba(X_test)[:, 1]

In [ ]:
print(classification_report(Y_test, y_pred_rf_smote))

In [ ]:
y_pred_threshold = (y_proba_rf >= 0.2).astype(int)

imbalance_results = pd.DataFrame([
    evaluate_model("Baseline DT", Y_test, y_pred_dt2, y_proba_dt),
    evaluate_model("DT + threshold 0.2", Y_test, y_pred_threshold, y_proba_dt),
    evaluate_model("DT + SMOTE", Y_test, y_pred_dt_smote, y_proba_dt_smote),
])

imbalance_results.round(3)

PART 6 - THE TIE IN

In [ ]:
attrition_rate = df.groupby('kmeans_clusters')['Attrition_Yes'].agg(
    Total='count',
    Positive_Class='sum',
    Attrition_Rate='mean'
).reset_index()

attrition_rate['Attrition_Rate_%'] = (attrition_rate['Attrition_Rate'] * 100).round(2)

print(attrition_rate[['kmeans_clusters', 'Total', 'Positive_Class', 'Attrition_Rate_%']])

Cluster 2 and Cluster 0 were the strongest predictors of my dataset.

Cluster 2 which represents Low Income Early Creer drifters has the highest rate of attrition of 33.08% amongst the clusters. It is the strongest positive predictor of the dataset.

Cluster 0 which represents Tenured High Level Professionals. This is the stongest negative predictor with an attrition rate of 5.86%.

It also implies that the clustering technique applied without any knowledge of the target variable produced clusters with such divergent attrition rates confirms that the features used capture genuine and meaningful patterns in employee turnover behaviour.

**PART 7 - CONCLUSION AND RECOMMENDATION**

We would deploy Logistic Regression because it had a performance than Decision Tree and Random Forest both in their baseline model and after imbalance handling. We would deploy Logistic Regression with threshold = 0.3 because it gave the best evaluation metric required for our IBM HR Employee Attrition.

We focus more on recall and ROC-AUC because we want to catch the actual leavers before they do and Logistic Regression with threshold of 0.3 provided the best recall value which is 0.532 and ROC-AUC OF 0.532.

If we had another month, we would try other algorithms like BOOSTING techniques.

2. We would do deep feature engineering

3. We would also do hyperparameter tuning

The hardest decision we made was choosing our value of our final K. Our elbow method gave us 4 while the silhouette score suggested 2. We did not pick 2 as the final K because it had broad personas and gave limited insights for the attrition while K=4 revealed actionable insights as to the cause of leaving and staying.

This decision taught the team that in applied data science, the right answer is not always the one with the highest score — it is the one that best serves the problem you are trying to solve.